# 🌹 Learning the ASCII Rose

This notebook rebuilds the idea behind `main.py` **from zero**, one small step at a time.
Every step prints real ASCII art, so you can see what each piece of maths does.

| Part | Idea | What it gives you |
|---|---|---|
| 1 | Characters as pixels | brightness → a character |
| 2 | sin & cos | drawing curves |
| 3 | Vectors (dot & cross product) | the maths behind lighting |
| 4 | A shaded ball | your first 3D-looking picture |
| 5 | Rotation | making things spin |
| 6 | Parametric surfaces | describing shapes with two "sliders" |
| 7 | The renderer (projection + z-buffer) | turning 3D points into a picture |
| 8 | The rose formula | the real thing |
| 9 | How it maps to `main.py` | reading the full program |

**How to use it:** pick the `.venv` kernel (top right in VS Code), then run the cells top to bottom with **Shift+Enter**.
Change numbers and re-run — breaking things is the fastest way to learn.

In [ ]:
import time
import numpy as np
from IPython.display import clear_output

## Part 1 — Characters as pixels

A screen is a grid of pixels. Here our "pixels" are **characters**. Some characters use a little ink (`.`),
some use a lot (`@`). If we line them up from least ink to most ink we get a **brightness ramp**:

```
" .:-=+*#%@"      dark  →  bright
```

So a brightness `b` between 0 and 1 picks a character: `RAMP[int(b * 9)]`. That is the *entire* trick of ASCII art.

One catch: a character cell is roughly **twice as tall as it is wide**. If we don't correct for that, circles come out as tall ovals.

In [ ]:
RAMP = " .:-=+*#%@"

def shade_char(b):
    """Brightness 0..1  ->  one character."""
    b = min(max(b, 0), 1)                     # clamp into 0..1
    return RAMP[int(b * (len(RAMP) - 1))]

# a brightness bar from 0 to 1
print("".join(shade_char(i / 59) for i in range(60)))

## Part 2 — sin & cos draw curves

As an angle `t` goes from 0 to 2π (a full turn), the point `(cos t, sin t)` walks around a circle of radius 1.
Multiply by `r` to change the radius:

$$x = r\cos t, \qquad y = r\sin t$$

This is called a **parametric** curve: one number (`t`) is turned into a position.
Nearly every shape in `main.py` is built out of this.

In [ ]:
def blank(W=60, H=30):
    return [[" "] * W for _ in range(H)]

def show(grid):
    print("\n".join("".join(row) for row in grid))

def plot(grid, x, y, ch="@", size=1.2):
    """Put a character at world position (x, y). (0, 0) is the centre of the grid."""
    H, W = len(grid), len(grid[0])
    col = int(W / 2 + x * (W / 2) / size)
    row = int(H / 2 - y * (W / 4) / size)     # W/4 instead of W/2: cells are ~2x taller
    if 0 <= row < H and 0 <= col < W:
        grid[row][col] = ch

g = blank()
for t in np.linspace(0, 2 * np.pi, 200):
    plot(g, np.cos(t), np.sin(t))
show(g)

**Try it:** let the radius grow while you go round several times and you get a **spiral** —
the backbone of the rose (the rose is a spiral whose "radius" is made of petals).

In [ ]:
g = blank()
for t in np.linspace(0, 6 * np.pi, 600):     # 3 full turns
    r = t / (6 * np.pi)                       # radius grows from 0 to 1
    plot(g, r * np.cos(t), r * np.sin(t), "*")
show(g)

## Part 3 — Vectors: the dot product and the cross product

A point or a direction in 3D is just three numbers `(x, y, z)`. You need exactly two operations on them.

**Dot product** `a · b = ax·bx + ay·by + az·bz`
For two directions of length 1 it tells you **how much they point the same way**:
`1` = same direction, `0` = at right angles, `-1` = opposite.

👉 This *is* lighting: a surface is bright when the direction it faces points toward the light.

**Cross product** `a × b` gives a new direction that is **at right angles to both** `a` and `b`.

👉 This is how we find which way a surface faces (its **normal**): take two directions that lie *along*
the surface; their cross product sticks straight *out* of it.

In [ ]:
up    = np.array([0, 0, 1])
side  = np.array([1, 0, 0])
diag  = np.array([1, 0, 1]) / np.sqrt(2)      # 45° between up and side (divide to make length 1)

print("up · up   =", up @ up)       # 1   same direction
print("up · diag =", round(up @ diag, 3))     # 0.707  partly the same
print("up · side =", up @ side)     # 0   right angles
print("up · -up  =", up @ -up)      # -1  opposite

print("side × forward =", np.cross(side, [0, 1, 0]))   # -> (0, 0, 1): straight up, out of the floor

## Part 4 — A shaded ball (your first 3D picture)

For every character cell we ask: *is this cell on the ball, and if so, how bright is it?*

1. Map the cell to coordinates `x, y` between -1 and 1.
2. If `x² + y² > 1` we're outside the circle → blank.
3. Otherwise the sphere's surface is at height `z = √(1 − x² − y²)` (Pythagoras).
4. On a sphere centred at the origin, the point **is** its own normal: `normal = (x, y, z)`.
5. Brightness = `normal · light` (Part 3). Negative means facing away → 0.
6. Brightness → character (Part 1).

Change `light` and watch the highlight move.

In [ ]:
def ball(light, W=60, H=30):
    light = np.array(light, float)
    light /= np.linalg.norm(light)            # make it length 1
    lines = []
    for row in range(H):
        line = ""
        for col in range(W):
            x = (col - W / 2) / (W / 2)
            y = (H / 2 - row) / (H / 2)
            if x * x + y * y > 1:
                line += " "
                continue
            z = np.sqrt(max(0.0, 1 - x * x - y * y))   # max(): rounding can dip just below 0
            normal = np.array([x, y, z])
            line += shade_char(normal @ light)
        lines.append(line)
    print("\n".join(lines))

ball(light=[-1, 1, 1])      # light from upper-left, in front

In [ ]:
ball(light=[1, -0.3, 0.4])  # light from the right and a bit below — try your own!

## Part 5 — Rotation

To turn a point by angle `a` around the vertical axis, mix its `x` and `y` with cos and sin:

$$x' = x\cos a - y\sin a, \qquad y' = x\sin a + y\cos a$$

Why? `(cos a, sin a)` is where the point `(1, 0)` lands after turning by `a` (Part 2),
and `(-sin a, cos a)` is where `(0, 1)` lands. Every other point is a mix of those two.

We use **two** rotations, exactly like `main.py`:
* `spin` — turn the object around its vertical axis (the animation),
* `tilt` — tip it toward the camera so we look slightly down on it.

**Coordinate convention (same as `main.py`):** `z` is up, `x` is right, and `y` points *away* from the camera,
so a smaller `y` means closer to you.

In [ ]:
def rotate(P, spin, tilt):
    """P is an array of shape (N, 3): N points, each (x, y, z)."""
    x, y, z = P[:, 0], P[:, 1], P[:, 2]
    # spin around the vertical (z) axis
    x, y = x * np.cos(spin) - y * np.sin(spin), x * np.sin(spin) + y * np.cos(spin)
    # tilt around the left-right (x) axis: same formula, applied to (y, z)
    y, z = y * np.cos(tilt) - z * np.sin(tilt), y * np.sin(tilt) + z * np.cos(tilt)
    return np.stack([x, y, z], axis=1)

p = np.array([[1.0, 0.0, 0.0]])
print("(1,0,0) spun 90° ->", rotate(p, np.pi / 2, 0).round(3))   # lands on (0, 1, 0)

> **Why numpy arrays?** `P` holds *all* points at once (shape `(N, 3)`). `P[:, 0]` is every x-coordinate.
> Writing `x * np.cos(spin)` multiplies a million numbers in one go — far faster than a Python `for` loop.
> That's why `main.py` has almost no loops and looks "dense".

## Part 6 — Parametric surfaces: shapes made from two sliders

A curve (Part 2) uses **one** number `t`. A *surface* uses **two**, say `u` and `v`.
Sweep both over a grid and you get a grid of 3D points covering the shape.

**Torus (a donut):** `u` goes around the big ring, `v` goes around the tube.

$$X = (R + r\cos v)\cos u,\quad Y = (R + r\cos v)\sin u,\quad Z = r\sin v$$

Look closely: it's just Part 2's circle, used twice.

**Normals from the grid:** neighbouring points in the `u` direction give one direction *along* the surface,
neighbours in the `v` direction give another. Their **cross product** (Part 3) points straight out.
`np.gradient` computes those neighbour differences for the whole grid at once.

In [ ]:
def normals(P):
    """P has shape (rows, cols, 3) — a grid of points. Returns unit normals, same shape."""
    along_u = np.gradient(P, axis=0)          # step to the next row
    along_v = np.gradient(P, axis=1)          # step to the next column
    n = np.cross(along_u, along_v)            # sticks out of the surface
    return n / (np.linalg.norm(n, axis=-1, keepdims=True) + 1e-12)   # make length 1

def torus(R=1.0, r=0.4, nu=300, nv=80):
    u, v = np.meshgrid(np.linspace(0, 2 * np.pi, nu), np.linspace(0, 2 * np.pi, nv), indexing="ij")
    P = np.stack([(R + r * np.cos(v)) * np.cos(u),
                  (R + r * np.cos(v)) * np.sin(u),
                  r * np.sin(v)], axis=-1)
    return P.reshape(-1, 3), normals(P).reshape(-1, 3)   # flatten grid -> list of points

P, N = torus()
print(P.shape, "points, each with a normal:", N.shape)

## Part 7 — The renderer: project, z-buffer, light

Now we turn a cloud of 3D points into a picture. This is the heart of `main.py` (`shade_scene`).

1. **Rotate** points and normals (Part 5).
2. **Light:** brightness = `normal · light` (Part 3), plus a little "ambient" light so shadows aren't pure black.
   A thin petal can be seen from either side, so if a normal faces away from the camera we flip it (**two-sided lighting**).
3. **Project:** drop the depth axis. Screen column comes from `x`, screen row from `z`.
4. **Z-buffer:** many points land in the same cell. Keep only the **nearest** one (smallest `y`).
   Trick: sort all points by distance, then keep the *first* point of each cell.
5. Brightness → character.

In [ ]:
LIGHT = np.array([-0.5, -0.7, 0.6])
LIGHT /= np.linalg.norm(LIGHT)                # from the upper-left, toward the camera

def render(P, N, spin=0.0, tilt=0.4, W=80, H=40, size=1.3, cz=0.0):
    """Return the ASCII picture as a string. `size`: world units from centre to edge. `cz`: height to centre on."""
    P = rotate(P, spin, tilt)
    N = rotate(N, spin, tilt)

    # 2. lighting (two-sided: flip normals that face away from the camera, i.e. have positive y)
    N = N * np.where(N[:, 1] > 0, -1.0, 1.0)[:, None]
    brightness = 0.1 + 0.9 * np.clip(N @ LIGHT, 0, 1)

    # 3. projection: world -> character cell
    sx = (W / 2) / size                        # characters per world unit, horizontally
    sy = sx / 2                                # half as many vertically (tall cells)
    col = (W / 2 + P[:, 0] * sx).astype(int)
    row = (H / 2 - (P[:, 2] - cz) * sy).astype(int)
    on_screen = (col >= 0) & (col < W) & (row >= 0) & (row < H)
    cell = (row * W + col)[on_screen]
    depth = P[on_screen, 1]
    brightness = brightness[on_screen]

    # 4. z-buffer: nearest first, then keep the first point of each cell
    order = np.argsort(depth)
    _, first = np.unique(cell[order], return_index=True)
    winners = order[first]

    # 5. brightness -> characters
    screen = np.full(W * H, " ")
    idx = (brightness[winners] * (len(RAMP) - 1)).astype(int)
    screen[cell[winners]] = np.array(list(RAMP))[idx]
    return "\n".join("".join(screen[r * W:(r + 1) * W]) for r in range(H))

print(render(P, N, spin=0.3, tilt=0.6, size=1.6))

### Make it spin
An animation is just: change the angle a little, redraw, repeat.
(`clear_output` wipes the previous frame. Interrupt the cell with the ■ button to stop early.)

In [ ]:
for frame in range(80):
    clear_output(wait=True)
    print(render(P, N, spin=frame * 0.08, tilt=0.6 + 0.3 * np.sin(frame * 0.05), size=1.6))
    time.sleep(0.04)

🎉 **That is the whole engine.** Everything from here on is *what shape* we feed into it.

## Part 8 — The rose formula

The rose is a parametric surface (Part 6) with two sliders:

* `t` — goes round and round in a **spiral** about 8.5 times (Part 2's spiral!). Outer petals come first, the core last.
* `x` — goes from 0 to 1: from the **centre** of the flower out to the **edge** of a petal.

Then, term by term:

| Term | Formula | What it does |
|---|---|---|
| `p` | `(π/2) · e^(−t / 8π) · bloom` | How far a petal leans outward from vertical. Outer petals (small `t`) lean a lot, the core stands upright. Multiplying by `bloom` opens/closes the flower. |
| `u` | `1 − ½(1.25(1 − mod(3.6t, 2π)/π)² − ¼)²` | `mod(3.6t, 2π)` restarts 3.6 times per turn — **each restart is one petal**. The curve gives each petal a rounded body and a pointed tip. |
| `y` | `1.95653 · x²(1.27689x − 1)² · sin p` | A small bend near the outer edge: the **outward curl** of each petal's lip. |
| `r` | `u(x sin p + y cos p)` | Distance from the centre axis. |
| height | `u(x cos p − y sin p)` | How high the point sits. (`r` and height together are just rotating `(x, y)` by angle `p` — Part 5 again!) |
| position | `(r cos t, r sin t, height)` | Part 2's circle with a changing radius. |

Nobody invents a formula like this in one go — it's a well-known shared formula, tuned by trial and error.
The skill is being able to *read* it, which you now can.

In [ ]:
def rose(bloom=0.9, petals_per_turn=3.6, nt=2400, nx=90):
    x, t = np.meshgrid(np.linspace(0, 1, nx), np.linspace(-2 * np.pi, 15 * np.pi, nt))
    p = (np.pi / 2) * np.exp(-t / (8 * np.pi)) * bloom
    u = 1 - 0.5 * (1.25 * (1 - np.mod(petals_per_turn * t, 2 * np.pi) / np.pi) ** 2 - 0.25) ** 2
    y = 1.95653 * x ** 2 * (1.27689 * x - 1) ** 2 * np.sin(p)
    r = u * (x * np.sin(p) + y * np.cos(p))
    height = u * (x * np.cos(p) - y * np.sin(p))
    P = np.stack([r * np.cos(t), r * np.sin(t), height], axis=-1)
    return P.reshape(-1, 3), normals(P).reshape(-1, 3)

RP, RN = rose(0.9)
print(render(RP, RN, spin=0.5, tilt=0.5, size=1.15, cz=0.35))

### Watch it bloom
Only `bloom` changes between frames — the rest of the formula stays the same.

In [ ]:
for frame in range(60):
    b = 0.2 + 0.75 * min(frame / 40, 1)       # open from 0.2 to 0.95, then hold
    P_, N_ = rose(b, nt=1600, nx=60)          # fewer points = faster
    clear_output(wait=True)
    print(render(P_, N_, spin=frame * 0.06, tilt=0.5, size=1.15, cz=0.35))
    print(f"bloom = {b:.2f}")
    time.sleep(0.03)

### Experiment: break the rose on purpose
Change one number at a time and predict what will happen *before* you run it:

* `petals_per_turn=2.2` → fewer, bigger petals. `5.0` → many thin ones.
* `bloom=0.4` → a closed bud. `bloom=1.2` → over-opened, petals droop.
* `tilt=1.2` → look straight down into the spiral. `tilt=0.0` → pure side view.

In [ ]:
P_, N_ = rose(bloom=0.9, petals_per_turn=2.2)
print(render(P_, N_, spin=0.5, tilt=1.1, size=1.15, cz=0.35))

## Part 9 — How this maps to `main.py`

You now know every core idea. `main.py` is the same pipeline with more polish:

| This notebook | `main.py` | Extra polish in `main.py` |
|---|---|---|
| `RAMP`, `shade_char` | `RAMP`, `build_ramp` | measures each character's real ink coverage to order the ramp |
| `normals` | `normals` | identical |
| `rose` | `rose_surface` | adds shading for petal rims and the dark inner core (`ao`) |
| `torus` | `stem_and_leaves`, `leaf`, `thorn` | tube stem, curved leaves with jagged edges, thorns |
| `rotate` | `rot` inside `shade_scene` | identical maths |
| `render` | `shade_scene` + `render` | 3 lights (key, fill, rim) + a specular shine, shadows between petals, colour per material |
| `print(...)` | `cells_to_image`, `bloom` | draws characters into a 1080×1350 image with a soft glow |
| animation loop | `main` | writes 168 frames to `rose.mp4` (ffmpeg) and `rose.gif` |

Plus decorations that aren't core maths: stars, rising sparkles, the "I love you" text, and the fade in/out.

**Suggested way to read `main.py`:** start at `render(f)` near the bottom and follow each function it calls.

## Exercises

1. **Colour:** in a terminal, `"\033[38;2;255;60;90m@\033[0m"` prints a red `@`. Make `render` colour each character by brightness.
2. **Specular shine:** add `0.5 * max(0, N · H)**30` to the brightness, where `H = normalize(LIGHT + [0, -1, 0])`. You'll get glossy highlights.
3. **A stem:** make a thin tube (a torus is a tube bent into a ring — straighten it!) below the rose, and render both point clouds together with `np.concatenate`.
4. **Your own shape:** replace the torus formula with something new, e.g. `Z = sin(3u)·cos(3v)` over a flat grid.

## Further reading

* **"Donut math: how donut.c works"** — Andy Sloane (a1k0n.net). The classic explanation of exactly this technique.
* **3Blue1Brown — "Essence of Linear Algebra"** (YouTube), episodes 1–4: vectors, dot products, rotations.
* **Desmos 3D** (desmos.com/3d) — paste the rose equations and drag sliders to see each term move.